In [1]:
import polars as pl

In [2]:
url = "https://gitlab.com/exploit-database/exploitdb/-/raw/main/files_exploits.csv?ref_type=heads"
df = pl.read_csv(url)

In [3]:
df.head()

id,file,description,date_published,author,type,platform,port,date_added,date_updated,verified,codes,tags,aliases,screenshot_url,application_url,source_url
i64,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str
16929,"""exploits/aix/dos/16929.rb""","""AIX Calendar Manager Service D…","""2010-11-11""","""Metasploit""","""dos""","""aix""",null,"""2010-11-11""","""2011-03-06""",1,"""CVE-2009-3699;OSVDB-58726""","""Metasploit Framework (MSF)""",null,null,null,"""http://aix.software.ibm.com/ai…"
19046,"""exploits/aix/dos/19046.txt""","""AppleShare IP Mail Server 5.0.…","""1999-10-15""","""Chris Wedgwood""","""dos""","""aix""",null,"""1999-10-15""","""2014-01-02""",1,"""CVE-1999-1015;OSVDB-5970""",null,null,null,null,"""https://www.securityfocus.com/…"
19049,"""exploits/aix/dos/19049.txt""","""BSDI 4.0 tcpmux / inetd - Cras…","""1998-04-07""","""Mark Schaefer""","""dos""","""aix""",null,"""1998-04-07""","""2014-01-02""",1,"""OSVDB-82889""",null,null,null,null,"""https://www.securityfocus.com/…"
33943,"""exploits/aix/dos/33943.txt""","""Flussonic Media Server 4.1.25 …","""2014-07-01""","""BGA Security""","""dos""","""aix""",8080,"""2014-07-01""","""2014-07-01""",0,"""OSVDB-108610;OSVDB-108609""",null,null,null,null,null
19418,"""exploits/aix/dos/19418.txt""","""IBM AIX 4.3.1 - 'adb' Denial o…","""1999-07-12""","""GZ Apple""","""dos""","""aix""",null,"""1999-07-12""","""2017-11-15""",1,"""OSVDB-83455""",null,null,null,null,"""https://www.securityfocus.com/…"


In [10]:
import re

# 处理codes列：优先匹配CVE编号，如果没有CVE编号，就用OSVDB编号。如果为空字符串，使用file代替
def process_codes(codes, file_path):
    if not codes or codes == "":
        # 如果codes为空，使用file路径作为标识符
        return file_path

    # 优先匹配CVE编号
    cve_match = re.search(r'CVE-\d{4}-\d+', codes)
    if cve_match:
        return cve_match.group(0)

    # 如果没有CVE编号，匹配OSVDB编号
    osvdb_match = re.search(r'OSVDB-\d+', codes)
    if osvdb_match:
        return osvdb_match.group(0)

    # 如果都没有匹配到，返回原始codes
    return codes

# 处理file列：提取文件名的数字，并拼接为exploit-db URL
def process_file(file_path):
    # 使用正则表达式提取路径中的数字
    match = re.search(r'/(\d+)\.', file_path)
    if match:
        exploit_id = match.group(1)
        return f"https://www.exploit-db.com/exploits/{exploit_id}"
    return file_path

# 应用转换 - 使用struct来同时访问多个列
df_processed = df.with_columns([
    pl.struct(["codes", "file"]).map_elements(
        lambda x: process_codes(x["codes"], x["file"]),
        return_dtype=pl.Utf8
    ).alias("cve_id"),
    pl.col("description").alias("description"),  # 保留description
    pl.col("file").map_elements(process_file, return_dtype=pl.Utf8).alias("github_url")
])

# 显示处理结果
df_processed.head()

id,file,description,date_published,author,type,platform,port,date_added,date_updated,verified,codes,tags,aliases,screenshot_url,application_url,source_url,cve_id,github_url
i64,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str
16929,"""exploits/aix/dos/16929.rb""","""AIX Calendar Manager Service D…","""2010-11-11""","""Metasploit""","""dos""","""aix""",null,"""2010-11-11""","""2011-03-06""",1,"""CVE-2009-3699;OSVDB-58726""","""Metasploit Framework (MSF)""",null,null,null,"""http://aix.software.ibm.com/ai…","""CVE-2009-3699""","""https://www.exploit-db.com/exp…"
19046,"""exploits/aix/dos/19046.txt""","""AppleShare IP Mail Server 5.0.…","""1999-10-15""","""Chris Wedgwood""","""dos""","""aix""",null,"""1999-10-15""","""2014-01-02""",1,"""CVE-1999-1015;OSVDB-5970""",null,null,null,null,"""https://www.securityfocus.com/…","""CVE-1999-1015""","""https://www.exploit-db.com/exp…"
19049,"""exploits/aix/dos/19049.txt""","""BSDI 4.0 tcpmux / inetd - Cras…","""1998-04-07""","""Mark Schaefer""","""dos""","""aix""",null,"""1998-04-07""","""2014-01-02""",1,"""OSVDB-82889""",null,null,null,null,"""https://www.securityfocus.com/…","""OSVDB-82889""","""https://www.exploit-db.com/exp…"
33943,"""exploits/aix/dos/33943.txt""","""Flussonic Media Server 4.1.25 …","""2014-07-01""","""BGA Security""","""dos""","""aix""",8080,"""2014-07-01""","""2014-07-01""",0,"""OSVDB-108610;OSVDB-108609""",null,null,null,null,null,"""OSVDB-108610""","""https://www.exploit-db.com/exp…"
19418,"""exploits/aix/dos/19418.txt""","""IBM AIX 4.3.1 - 'adb' Denial o…","""1999-07-12""","""GZ Apple""","""dos""","""aix""",null,"""1999-07-12""","""2017-11-15""",1,"""OSVDB-83455""",null,null,null,null,"""https://www.securityfocus.com/…","""OSVDB-83455""","""https://www.exploit-db.com/exp…"


In [6]:
len(df_processed)  # 查看处理后的数据行数

46942

In [11]:
# 查询 cve_id 和 github_url是否重复
df_processed.select([
    pl.col("cve_id"),
    pl.col("github_url"),
    pl.count().alias("count")
]).group_by(["cve_id", "github_url"]).agg([
    pl.count().alias("count")
]).filter(pl.col("count") > 1).sort(pl.col("count"))

/tmp/ipykernel_269240/2496911895.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("count")
/tmp/ipykernel_269240/2496911895.py:7: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("count")


cve_id,github_url,count
str,str,u32
"""CVE-2018-9445""","""https://www.exploit-db.com/exp…",2
"""exploits/windows/local/46962.p…","""https://www.exploit-db.com/exp…",2
"""exploits/windows/dos/46845.py""","""https://www.exploit-db.com/exp…",2
"""OVE-20160718-0003""","""https://www.exploit-db.com/exp…",2
"""CVE-2017-2476""","""https://www.exploit-db.com/exp…",2
…,…,…
"""CVE-2018-8736""","""https://www.exploit-db.com/exp…",3
"""OSVDB-120224""","""https://www.exploit-db.com/exp…",3
"""OVE-20170126-0002""","""https://www.exploit-db.com/exp…",3
